# Sprint 54 Task 1 — Judge Fault Detection

Kaggle notebook conversion of `Sprint54_Task1_Judge_Fault_Detection.py`. It uses the shared `sprint54_judge_common.py` utility unchanged and the Sprint 51 `gold_set_labeled.csv` dataset.

**Before running:** enable a GPU accelerator (T4 x2 is preferred if available), enable Internet so Hugging Face can download Qwen3-8B, and attach a Kaggle dataset containing `gold_set_labeled.csv` and `sprint54_judge_common.py` (or upload these files to `/kaggle/working`). Run the setup cell, review the configuration, then run the experiment cell. The model load can take several minutes.

In [ ]:
from pathlib import Path
import argparse, json, re, string
import pandas as pd
import importlib.util, subprocess, sys, glob, os
if importlib.util.find_spec("unsloth") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])


# Locate input files whether attached as a Kaggle Dataset or uploaded to /kaggle/working.
def find_file(filename):
    candidates = [Path('/kaggle/working') / filename]
    candidates += [Path(p) for p in glob.glob('/kaggle/input/**/' + filename, recursive=True)]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(f'Could not find {filename}. Attach it as a Kaggle Dataset or upload it to /kaggle/working.')

# DATA_PATH = str(find_file('gold_set_labeled.csv'))
DATA_PATH = str(find_file('fault_cases.csv'))
COMMON_PATH = find_file('sprint54_judge_common.py')
sys.path.insert(0, str(COMMON_PATH.parent))
OUTPUT_DIR = '/kaggle/working/'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Configuration: edit these before running the experiment.
DIRECTION = 'EN_CMN'
MAX_EXAMPLES = 50
DRY_RUN = False
DRY_RUN_N = 3
print('Dataset:', DATA_PATH)
print('Shared utility:', COMMON_PATH)
print('Output:', OUTPUT_DIR)
print('GPU available:', __import__('torch').cuda.is_available())

## Experiment configuration

For the first run, set `DRY_RUN = True` and `DRY_RUN_N = 3` to validate the pipeline. This still loads the Qwen model. After reviewing outputs, set `DRY_RUN = False` for the full run. Results are saved under `/kaggle/working/` and can be downloaded from the notebook output panel.

In [ ]:
# Provide command-line arguments expected by the original experiment logic.
## Experiment configuration

# For the first run, set `DRY_RUN = True` and `DRY_RUN_N = 3` to validate the pipeline. 
# This still loads the Qwen model. After reviewing outputs, set `DRY_RUN = False` for the full run. 
# Results are saved under `/kaggle/working/` and can be downloaded from the notebook output panel.

argv = ['Sprint54_Task1_Judge_Fault_Detection.py', '--data', DATA_PATH, '--output', OUTPUT_DIR, '--direction', DIRECTION, '--max-examples', str(MAX_EXAMPLES)]

if DRY_RUN:
    argv += ['--dry-run', '--dry-run-n', str(DRY_RUN_N)]

sys.argv = argv
print('Arguments:', ' '.join(argv))

In [ ]:
"""Sprint 54 Task 1: measure controlled translation-fault detection by the P1 judge.

Run after installing the same Unsloth/PyTorch stack used in Sprint 53.
Example:
  python Sprint54_Task1_Judge_Fault_Detection.py --data gold_set_labeled.csv --output sprint54_task1_outputs
"""


from sprint54_judge_common import (
    P1Judge, 
    environment_metadata, 
    MODEL_NAME as COMMON_MODEL_NAME, 
    MAX_SEQ_LENGTH,
    MAX_NEW_TOKENS, 
    LOAD_IN_4BIT, 
    PROMPT_ID,
)

MODEL_NAME = "Qwen/Qwen3-8B"
# MODEL_NAME = "zai-org/GLM-4-9B-0414"

# Match the fault_type values in the pre-generated fault_cases.csv.
EXPECTATIONS = {
    "untranslated_english": 7.0,
    "negation_error": 5.0,
    "entity_error": 4.0,
    "number_error": 4.0,
    "dropped_clause": 3.0,
    "truncation": 3.0,
    "punctuation_control": 0.0,
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="Path to the pre-generated fault_cases.csv")
    ap.add_argument("--output", default="sprint54_task1_outputs")
    ap.add_argument("--direction", default="EN_CMN")
    ap.add_argument("--high-score-threshold", type=float, default=8.0)
    ap.add_argument("--max-examples", type=int, default=500)
    ap.add_argument("--dry-run", action="store_true")
    ap.add_argument("--dry-run-n", type=int, default=3)
    args = ap.parse_args()

    out = Path(args.output)
    out.mkdir(parents=True, exist_ok=True)

    # Load the already-generated fault cases. Do NOT run MUTATORS again.
    cases = pd.read_csv(args.data)

    required_columns = {
        "sample_id",
        "fault_type",
        "status",
        "source_en",
        "original_candidate",
        "modified_candidate",
        "overall_gold",
    }
    missing = required_columns - set(cases.columns)
    if missing:
        raise ValueError(f"Input CSV is missing required columns: {sorted(missing)}")

    # Only evaluate successfully generated mutations.
    # SKIPPED cases remain in the final output but are not sent to the judge.
    cases["fault_type"] = cases["fault_type"].astype(str)
    cases["status"] = cases["status"].astype(str).str.upper()

    unknown_faults = set(cases["fault_type"].unique()) - set(EXPECTATIONS)
    if unknown_faults:
        raise ValueError(f"Unknown fault types in CSV: {sorted(unknown_faults)}")

    # The supplied file was created using overall_gold > 8.
    # Reapply the requested high-score filter for safety.
    cases["overall_gold"] = pd.to_numeric(cases["overall_gold"], errors="coerce")
    cases = cases[cases["overall_gold"] > args.high_score_threshold].copy()

    # Avoid silently including cases for another translation direction,
    # if the CSV has a direction column.
    if "direction" in cases.columns:
        cases = cases[
            cases["direction"].isna()
            | cases["direction"].astype(str).eq(args.direction)
        ].copy()

    # Keep the original row count per fault type balanced as much as possible.
    # max-examples applies to unique source examples, not mutation rows.
    unique_ids = cases["sample_id"].drop_duplicates().head(args.max_examples)
    cases = cases[cases["sample_id"].isin(unique_ids)].copy()

    if args.dry_run:
        selected_ids = cases["sample_id"].drop_duplicates().head(args.dry_run_n)
        cases = cases[cases["sample_id"].isin(selected_ids)].copy()

    if cases.empty:
        raise ValueError(
            f"No fault cases found for direction {args.direction} "
            f"and overall_gold > {args.high_score_threshold}"
        )

    config = {
        "task": "Sprint54_Task1",
        "model_name": MODEL_NAME,
        "prompt_id": PROMPT_ID,
        "temperature": 0.0,
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "load_in_4bit": LOAD_IN_4BIT,
        "direction": args.direction,
        "high_score_threshold": args.high_score_threshold,
        "max_examples": args.max_examples,
        "input_type": "pre-generated fault_cases.csv",
        "expectations": EXPECTATIONS,
    }

    (out / "experiment_config.json").write_text(
        json.dumps(
            {
                **environment_metadata(config, args.data),
                "input_fault_rows": len(cases),
                "unique_source_examples": int(cases["sample_id"].nunique()),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    judge = P1Judge()
    results = []

    for idx, (_, row) in enumerate(cases.iterrows()):
        fault = row["fault_type"]
        expected_drop = EXPECTATIONS[fault]

        # Preserve pre-generated skips. Do not count them as successful tests.
        if row["status"] != "MUTATED" or pd.isna(row["modified_candidate"]):
            item = row.to_dict()
            item.update({
                "input_status": row["status"],
                "status": "SKIPPED",
                "skip_reason": row.get("skip_reason", "Mutation was not generated"),
                "original_judge_score": None,
                "modified_judge_score": None,
                "score_drop": None,
                "expected_minimum_drop": expected_drop,
                "threshold_met": None,
                "judge_feedback": None,
            })
            results.append(item)
            print(f"{row['sample_id']} | {fault}: SKIPPED")
            continue

        source = str(row["source_en"])
        original = str(row["original_candidate"])
        modified = str(row["modified_candidate"])

        # Evaluate both candidates using the same judge configuration.
        # This ensures score_drop is based on fresh scores from this run.
        original_result = judge.evaluate(
            source, original, temperature=0.0, seed=5400 + idx
        )
        modified_result = judge.evaluate(
            source, modified, temperature=0.0, seed=6400 + idx
        )

        item = row.to_dict()
        item.update({
            "input_status": row["status"],
            "original_judge_score": original_result.get("score"),
            "modified_judge_score": modified_result.get("score"),
            "original_parse_ok": original_result.get("parse_ok"),
            "modified_parse_ok": modified_result.get("parse_ok"),
            "expected_minimum_drop": expected_drop,
            "judge_feedback": modified_result.get("feedback"),
            "threshold_met": None,
            "score_drop": None,
            "status": "SCORED",
            "skip_reason": "",
        })

        if not original_result.get("parse_ok") or not modified_result.get("parse_ok"):
            item["status"] = "UNSCORABLE"
            item["skip_reason"] = (
                f"Original parse error: {original_result.get('error', '')}; "
                f"modified parse error: {modified_result.get('error', '')}"
            )
        else:
            drop = float(original_result["score"]) - float(modified_result["score"])
            item["score_drop"] = round(drop, 3)

            if fault == "punctuation_control":
                # Control should preserve the score, not trigger a large drop.
                # A small 1-point tolerance is used here.
                item["threshold_met"] = abs(drop) <= 1.0
            else:
                item["threshold_met"] = drop >= expected_drop

        results.append(item)
        print(
            f"{row['sample_id']} | {fault}: {item['status']} "
            f"| original={item['original_judge_score']} "
            f"| modified={item['modified_judge_score']} "
            f"| drop={item['score_drop']}"
        )

    detail = pd.DataFrame(results)
    detail.to_csv(out / "fault_detection_results.csv", index=False)

    summaries = []
    for fault in EXPECTATIONS:
        group = detail[detail["fault_type"] == fault]
        if group.empty:
            continue

        scored = group[group["status"] == "SCORED"]
        skipped = int((group["status"] == "SKIPPED").sum())
        unscorable = int((group["status"] == "UNSCORABLE").sum())
        expected = EXPECTATIONS[fault]

        pass_rate = (
            float(scored["threshold_met"].fillna(False).mean())
            if len(scored)
            else None
        )

        summaries.append({
            "fault_type": fault,
            "expected_minimum_drop": expected,
            "attempted": len(group),
            "scored": len(scored),
            "skipped": skipped,
            "unscorable": unscorable,
            "mean_score_drop": (
                round(scored["score_drop"].mean(), 3)
                if len(scored) else None
            ),
            "median_score_drop": (
                round(scored["score_drop"].median(), 3)
                if len(scored) else None
            ),
            "threshold_met_count": (
                int(scored["threshold_met"].fillna(False).sum())
                if len(scored) else 0
            ),
            "threshold_met_rate": (
                round(pass_rate, 3) if pass_rate is not None else None
            ),
            "finding_status": (
                "INSUFFICIENT_DATA"
                if len(scored) < 3
                else (
                    "MEETS_EXPECTATION_ON_TESTED_CASES"
                    if pass_rate >= 0.8
                    else "NOT_RELIABLY_DETECTED_ON_TESTED_CASES"
                )
            ),
        })

    summary = pd.DataFrame(summaries)
    summary.to_csv(out / "fault_detection_summary.csv", index=False)

    lines = [
        "# Sprint 54 Task 1 — Judge Fault Detection Finding",
        "",
        f"Judge: {MODEL_NAME}; prompt: {PROMPT_ID}; temperature: 0.0.",
        f"Input file: `{args.data}`.",
        f"Unique source examples: {cases['sample_id'].nunique()}.",
        f"Fault-case rows evaluated or retained: {len(cases)}.",
        "",
        "Only pre-generated mutations marked MUTATED were scored. Pre-generated "
        "SKIPPED cases remain in the results and are not counted as passes.",
        "Fault thresholds are assessed only on successfully mutated and scored cases.",
        "For punctuation control, a score-drop magnitude of at most 1 point is "
        "treated as preserving the score.",
        "",
        "## Results",
        "",
        "| Fault | Expected drop | Scored | Skipped | Unscorable | Mean drop | Threshold met | Finding |",
        "|---|---:|---:|---:|---:|---:|---:|---|",
    ]

    for _, r in summary.iterrows():
        mean_drop = (
            r["mean_score_drop"]
            if pd.notna(r["mean_score_drop"])
            else "N/A"
        )
        rate = f"{r['threshold_met_count']}/{r['scored']}"
        lines.append(
            f"| {r['fault_type']} | {r['expected_minimum_drop']:g} "
            f"| {r['scored']} | {r['skipped']} | {r['unscorable']} "
            f"| {mean_drop} | {rate} | {r['finding_status']} |"
        )

    lines += [
        "",
        "## Interpretation",
        "",
        "Results apply to the selected high-scoring examples and the specific "
        "mutations present in the input CSV. A threshold pass rate of at least "
        "80% with at least three scored cases is labeled as meeting expectations "
        "on tested cases; it is not a population-level reliability guarantee.",
        "Review `fault_detection_results.csv` to inspect each original and "
        "modified candidate. Mutation quality, especially for negation, entity, "
        "number, and dropped-clause errors, should be checked before drawing "
        "strong conclusions.",
        "",
    ]

    (out / "findings.md").write_text("\n".join(lines), encoding="utf-8")
    print("\nSaved outputs to", out.resolve())


main()



## Output files

Review `findings.md` first, then the CSV files for detailed results. Task 1: inspect `fault_cases.csv`; skipped/unscorable mutations are not passes. Task 2: compare within-example score ranges at temperature 0.0 vs 0.2, and treat sample-size calculations as planning estimates rather than a definitive power analysis.